# W2-D2: Root Cause Analysis — Graph + Temporal + Incident Retrieval

Pipeline:
1. Load data (alerts, service graph, incident history)
2. **Import clusters từ W2-D1** (`cluster_summary.json`)
3. Build directed service graph với networkx
4. Score mỗi service: graph terminal score + PageRank + temporal
5. Retrieve similar historical incidents (keyword heuristic)
6. Classify root cause class via kNN top-1
7. Validate output + fallback
8. Write `results/rca_output.json`

In [1]:
import os
os.chdir(r'C:\Users\husky\Downloads\life\aiops-g4-HungNguyen\w2\d2')
os.makedirs('results', exist_ok=True)
print('Dependencies ready')

Dependencies ready


## Step 1: Load All Data

In [2]:
import json
import networkx as nx
from datetime import datetime

alerts = []
with open('dataset/alerts_sample.jsonl') as f:
    for line in f:
        line = line.strip()
        if line:
            alerts.append(json.loads(line))
with open('dataset/services.json', encoding='utf-8') as f:
    services_data = json.load(f)
with open('dataset/incidents_history.json', encoding='utf-8') as f:
    history = json.load(f)

print(f'Alerts raw      : {len(alerts)}')
print(f'Services        : {len(services_data["services"])}')
print(f'Incident history: {len(history["incidents"])}')

Alerts raw      : 20
Services        : 10
Incident history: 29


## Step 2: Import Clusters từ W2-D1

In [3]:
with open('dataset/cluster_summary.json') as f:
    cluster_summary = json.load(f)

print(f'W2-D1 summary  : {cluster_summary["input_alerts"]} alerts → {cluster_summary["output_clusters"]} cluster(s)')
print(f'Reduction ratio: {cluster_summary["reduction_ratio"]}')
print()

alert_by_id = {a['id']: a for a in alerts}
clusters = []
for c in cluster_summary['clusters']:
    cluster_alerts = [alert_by_id[aid] for aid in c['alert_ids'] if aid in alert_by_id]
    clusters.append({
        'cluster_id'  : c['cluster_id'],
        'services'    : c['services'],
        'alerts'      : cluster_alerts,
        'alert_count' : c['alert_count'],
        'severity_max': c.get('max_severity', 'warn'),
        'first_ts'    : c['time_range'][0],
        'last_ts'     : c['time_range'][1],
        'fingerprints': c.get('fingerprints', []),
    })

print(f'Clusters imported: {len(clusters)}')
for c in clusters:
    print(f"  {c['cluster_id']}: {c['alert_count']} alerts, sev={c['severity_max']}")
    print(f"  Services : {c['services']}")
    print(f"  Window   : {c['first_ts']} → {c['last_ts']}")

W2-D1 summary  : 20 alerts → 5 cluster(s)
Reduction ratio: 0.75

Clusters imported: 5
  c-000-000: 15 alerts, sev=crit
  Services : ['checkout-svc', 'edge-lb', 'payment-svc']
  Window   : 2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z
  c-000-001: 1 alerts, sev=warn
  Services : ['cart-svc']
  Window   : 2026-06-12T09:43:32Z → 2026-06-12T09:43:32Z
  c-000-002: 2 alerts, sev=crit
  Services : ['notification-svc']
  Window   : 2026-06-12T09:43:50Z → 2026-06-12T09:48:25Z
  c-000-003: 1 alerts, sev=warn
  Services : ['recommender-svc']
  Window   : 2026-06-12T09:45:10Z → 2026-06-12T09:45:10Z
  c-000-004: 1 alerts, sev=warn
  Services : ['search-svc']
  Window   : 2026-06-12T09:46:50Z → 2026-06-12T09:46:50Z


## Step 3: Build Service Graph

In [4]:
G = nx.DiGraph()
for svc in services_data['services']:
    G.add_node(svc['name'], type='service', tier=svc['tier'],
               criticality=svc['criticality'], team=svc['team'])
for store in services_data['stores']:
    G.add_node(store['name'], type='store', criticality=store['criticality'])
for edge in services_data['edges']:
    G.add_edge(edge['from'], edge['to'], type=edge['type'])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print()
print('Call graph (A → B means A calls B):')
for u, v, data in G.edges(data=True):
    print(f'  {u:20s} --> {v:20s} [{data["type"]}]')

Graph: 14 nodes, 17 edges

Call graph (A → B means A calls B):
  edge-lb              --> auth-svc             [http]
  edge-lb              --> catalog-svc          [http]
  edge-lb              --> search-svc           [http]
  edge-lb              --> checkout-svc         [http]
  checkout-svc         --> cart-svc             [http]
  checkout-svc         --> payment-svc          [http]
  checkout-svc         --> inventory-svc        [http]
  checkout-svc         --> notification-svc     [kafka]
  payment-svc          --> payments-db          [postgres]
  cart-svc             --> cart-redis           [redis]
  cart-svc             --> catalog-svc          [http]
  catalog-svc          --> catalog-db           [postgres]
  catalog-svc          --> recommender-svc      [http]
  recommender-svc      --> catalog-db           [postgres]
  inventory-svc        --> catalog-db           [postgres]
  notification-svc     --> kafka-events         [kafka]
  search-svc           --> catalog-db 

## Step 4: Graph + Temporal Scoring

**Score**:
- Terminal score = `1/(1+out_degree)` trong alerting subgraph
- PageRank trên original subgraph
- Combined graph = 50% terminal + 50% PR
- Final = **60% graph + 40% temporal**

In [5]:
def score_by_graph(cluster, G):
    services = cluster['services']
    subgraph_nodes = [s for s in services if s in G.nodes
                      and G.nodes[s].get('type') != 'store']
    subgraph = G.subgraph(subgraph_nodes).copy()
    if not subgraph_nodes:
        return {s: 0.5 for s in services}
    terminal_scores = {n: 1.0/(1+subgraph.out_degree(n)) for n in subgraph_nodes}
    try:
        pr = nx.pagerank(subgraph, alpha=0.85, max_iter=200)
    except Exception:
        pr = terminal_scores.copy()
    max_pr = max(pr.values()) or 1
    pr_norm = {k: v/max_pr for k, v in pr.items()}
    combined = {n: 0.5*terminal_scores[n] + 0.5*pr_norm.get(n, 0.3) for n in subgraph_nodes}
    max_c = max(combined.values()) or 1
    normalized = {k: v/max_c for k, v in combined.items()}
    for s in services:
        if s not in normalized:
            normalized[s] = 0.3
    return normalized

def score_by_timestamp(cluster):
    services = cluster['services']
    earliest = {}
    for a in cluster['alerts']:
        svc = a['service']
        if svc not in earliest or a['ts'] < earliest[svc]:
            earliest[svc] = a['ts']
    if len(earliest) <= 1:
        return {s: 1.0 for s in services}
    ts_min = min(earliest.values())
    ts_max = max(earliest.values())
    scores = {}
    for svc in services:
        if svc not in earliest:
            scores[svc] = 0.5; continue
        t0 = datetime.fromisoformat(ts_min.replace('Z','+00:00')).timestamp()
        t1 = datetime.fromisoformat(ts_max.replace('Z','+00:00')).timestamp()
        tc = datetime.fromisoformat(earliest[svc].replace('Z','+00:00')).timestamp()
        scores[svc] = 1.0 if t1==t0 else 1.0 - (tc-t0)/(t1-t0)
    return scores

def rca_combined(cluster, G, w_graph=0.6, w_time=0.4):
    g = score_by_graph(cluster, G)
    t = score_by_timestamp(cluster)
    combined = {svc: round(w_graph*g.get(svc,0.3) + w_time*t.get(svc,0.5), 4)
                for svc in cluster['services']}
    return sorted(combined.items(), key=lambda x: x[1], reverse=True)

# Test on first cluster
c = clusters[0]
ranked = rca_combined(c, G)
g_scores = score_by_graph(c, G)
t_scores = score_by_timestamp(c)

print(f'Cluster: {c["cluster_id"]}')
print(f'{"Service":22s}  {"combined":>10}  {"graph":>8}  {"temporal":>10}')
print('-'*58)
for svc, score in ranked:
    print(f'{svc:22s}  {score:>10.4f}  {g_scores.get(svc,0):>8.3f}  {t_scores.get(svc,0):>10.3f}')

Cluster: c-000-000
Service                   combined     graph    temporal
----------------------------------------------------------
payment-svc                 1.0000     1.000       1.000
checkout-svc                0.5279     0.610       0.405
edge-lb                     0.2666     0.444       0.000


## Step 5: Retrieve Similar Incidents

In [6]:
def retrieve_similar_incidents(cluster, history, top_k=3):
    cluster_services = set(cluster['services'])
    cluster_sev = cluster['severity_max']
    scored = []
    for inc in history['incidents']:
        score = 0.0
        if inc['root_cause_service'] in cluster_services: score += 0.4
        overlap = cluster_services & set(inc['services_involved'])
        score += min(0.2*len(overlap), 0.4)
        inc_sev = 'crit' if inc['severity']=='critical' else 'warn'
        if inc_sev == cluster_sev: score += 0.2
        if score >= 0.2:
            scored.append((inc, round(score,2)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

similar = retrieve_similar_incidents(clusters[0], history)
print('Top similar incidents:')
for inc, score in similar:
    print(f"  [{score:.2f}] {inc['id']} — {inc['root_cause_class']} on {inc['root_cause_service']}")
    print(f"         {inc['summary'][:90]}")
    print(f"         → {inc['remediation'][:80]}")
    print()

Top similar incidents:
  [1.00] INC-2025-11-08 — connection_pool_exhaustion on payment-svc
         Payment-svc v3.2 deploy at 09:42 leak DB pool. Pool 50/50 used trong 5 phút. Downstream ch
         → Rollback to v3.1. Scale pool 50 → 100 cushion. Add pool monitor alert > 80%.

  [1.00] INC-2026-03-20 — ddos on edge-lb
         Volumetric DDoS 5x normal traffic. Edge-lb saturate, all upstream visible degraded.
         → WAF rate-limit + Cloudflare proxy. Geographic rule.

  [0.80] INC-2025-09-05 — connection_pool_exhaustion on payment-svc
         Payment timeout 100%. Deploy v2.6 leak DB connection — pool from 50 → 50 hold, never retur
         → Rollback v2.6. Add max_idle_time + leak detection. Increase pool 50 → 100 cushio



## Step 6: Classify + Full Pipeline + Write Output

In [8]:
VALID_CLASSES = {
    'connection_pool_exhaustion','slow_query','memory_leak','rebalance_storm',
    'deadlock','network_partition','bad_deploy','config_push','tls_expiry','ddos','other'
}

def classify_from_history(similar_incidents):
    if not similar_incidents:
        return 'other', ['Investigate manually']
    best_inc, _ = similar_incidents[0]
    cls = best_inc['root_cause_class'] if best_inc['root_cause_class'] in VALID_CLASSES else 'other'
    return cls, [best_inc['remediation']]

def validate_output(result, cluster_services):
    issues = []
    if result['root_cause'] not in cluster_services: issues.append('root_cause not in cluster')
    if result['class'] not in VALID_CLASSES: issues.append('invalid class')
    if not (0.0 <= result['confidence'] <= 1.0): issues.append('confidence out of range')
    if not result['actions']: issues.append('empty actions')
    return issues

def run_rca_for_cluster(cluster, G, history):
    ranked = rca_combined(cluster, G)
    graph_top3 = [[svc, score] for svc, score in ranked[:3]]
    root_cause = ranked[0][0] if ranked else cluster['services'][0]
    confidence = ranked[0][1] if ranked else 0.3
    similar = retrieve_similar_incidents(cluster, history, top_k=3)
    similar_ids = [inc['id'] for inc, _ in similar]
    root_class, actions = classify_from_history(similar)
    top_inc = similar[0][0] if similar else None
    reasoning = (
        f"Graph RCA ranked {root_cause} top (score {confidence:.2f}). "
        f"Terminal in alerting subgraph + earliest alert. "
    )
    if top_inc:
        reasoning += f"Closest match: {top_inc['id']} — '{top_inc['summary'][:80]}'. Class: {root_class}."
    result = {
        'cluster_id': cluster['cluster_id'],
        'graph_top3': graph_top3,
        'root_cause': root_cause,
        'class': root_class,
        'confidence': round(confidence, 4),
        'actions': actions,
        'reasoning': reasoning,
        'similar_incidents': similar_ids,
        'method': 'graph+retrieval',
    }
    issues = validate_output(result, set(cluster['services']))
    if issues:
        print(f'  ⚠ Validation issues: {issues} — fallback')
        result.update({'root_cause': cluster['services'][0], 'class': 'other',
                       'actions': ['Investigate manually'], 'method': 'graph-only-fallback'})
    return result

all_results = []
for cluster in clusters:
    print(f"=== {cluster['cluster_id']} ===")
    r = run_rca_for_cluster(cluster, G, history)
    all_results.append(r)
    print(f"  Root cause : {r['root_cause']}")
    print(f"  Class      : {r['class']}")
    print(f"  Confidence : {r['confidence']}")
    print(f"  Top-3      : {r['graph_top3']}")
    print(f"  Similar    : {r['similar_incidents']}")
    print(f"  Actions    : {r['actions']}")
    print()

output = {'clusters_analyzed': len(clusters), 'results': all_results}
with open('results/rca_output.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(' Written to results/rca_output.json')
print()
print(json.dumps(output, indent=2, ensure_ascii=False))

=== c-000-000 ===
  Root cause : payment-svc
  Class      : connection_pool_exhaustion
  Confidence : 1.0
  Top-3      : [['payment-svc', 1.0], ['checkout-svc', 0.5279], ['edge-lb', 0.2666]]
  Similar    : ['INC-2025-11-08', 'INC-2026-03-20', 'INC-2025-09-05']
  Actions    : ['Rollback to v3.1. Scale pool 50 → 100 cushion. Add pool monitor alert > 80%.']

=== c-000-001 ===
  Root cause : cart-svc
  Class      : other
  Confidence : 1.0
  Top-3      : [['cart-svc', 1.0]]
  Similar    : ['INC-2025-07-19', 'INC-2026-02-22', 'INC-2025-06-12']
  Actions    : ['Tăng maxmemory 4GB → 8GB. Switch policy → volatile-lru. Set TTL trên transient keys.']

=== c-000-002 ===
  Root cause : notification-svc
  Class      : other
  Confidence : 1.0
  Top-3      : [['notification-svc', 1.0]]
  Similar    : ['INC-2026-02-08', 'INC-2026-05-18', 'INC-2025-07-04']
  Actions    : ['Failover to backup provider. Multi-provider routing.']

=== c-000-003 ===
  Root cause : recommender-svc
  Class      : memory_lea